##  Configuración del entorno

En una **terminal Miniconda** con el entorno `practica_env` activado:

```bash
conda activate practica_env
conda install -c conda-forge pandas numpy openpyxl matplotlib ipykernel
```

Elige en VS Code **Select Kernel → Python Environments → practica_env**. 

In [ ]:
# Librerías y confirmación del intérprete.
import sys
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

import pandas as pd
import numpy as np
from openpyxl import load_workbook

print("Intérprete de Python:", sys.executable)
print("Pandas:", pd.__version__)
print("Directorio desde el que ejecuta Jupyter:", Path.cwd())

Intérprete de Python: c:\Users\Florencia\miniconda3\envs\practica_env\python.exe
Pandas: 2.3.3
Directorio desde el que ejecuta Jupyter: c:\Users\Florencia\Desktop\Modulo 3\Practica modulo III\notebooks


In [ ]:
#  Encontrar automáticamente la raíz del proyecto y el archivo ZIP.
# Funciona si VS Code inicia Jupyter desde la carpeta raíz o desde notebooks/.
cwd = Path.cwd().resolve()
if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "notebooks").is_dir():
    PROJECT_ROOT = cwd
else:
    raise FileNotFoundError(
        f"No encuentro la carpeta 'notebooks' desde {cwd}. "
        "Abre la carpeta raíz del proyecto en VS Code y vuelve a ejecutar."
    )

ZIP_CANDIDATES = [
    PROJECT_ROOT / "data" / "raw" / "descarga_completa.zip", # Ubicación recomendada
    PROJECT_ROOT / "notebooks" / "descarga_completa.zip",   # Ubicación actual de tu captura
    PROJECT_ROOT / "descarga_completa.zip",                 # Alternativa válida
]
ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.is_file()), None)
if ZIP_PATH is None:
    rutas = "\n".join(str(p) for p in ZIP_CANDIDATES)
    raise FileNotFoundError(f"No encuentro descarga_completa.zip. Rutas examinadas:\n{rutas}")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR / "tabla_base_censo2024.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("ZIP seleccionado:", ZIP_PATH)
print(f"Tamaño ZIP: {ZIP_PATH.stat().st_size / (1024**2):.1f} MiB")
print("CSV de salida:", OUTPUT_PATH)

Raíz del proyecto: C:\Users\Florencia\Desktop\Modulo 3\Practica modulo III
ZIP seleccionado: C:\Users\Florencia\Desktop\Modulo 3\Practica modulo III\data\raw\descarga_completa.zip
Tamaño ZIP: 140.7 MiB
CSV de salida: C:\Users\Florencia\Desktop\Modulo 3\Practica modulo III\data\processed\tabla_base_censo2024.csv


##  Metodología CRISP-DM
### Fase de entendimiento del negocio

- **Necesidad institucional:** reconocer diferencias territoriales para facilitar el estudio de municipios/TIOC con características similares.
- **Resultado esperado:** datos integrados, estadísticas descriptivas, perfil de calidad y tabla minable en CSV.
- **Criterio de éxito técnico:** 3 libros integrados, 1 fila por territorio, cobertura completa de las claves, coherencia numérica y transformaciones documentadas.
- **Accesibilidad y presupuesto:** los datos del INE son públicos; hardware, tiempo y responsabilidades deben ser acordados por los cinco integrantes.
- **Riesgos:** encabezados Excel de varias filas, años censales distintos en un mismo libro, subtotales a distintos niveles y denominadores distintos en cada indicador.
- **Implementación:** la segmentación sería una herramienta de exploración; las decisiones institucionales requerirían validaciones adicionales.

### Fase de entendimiento de los datos — integración

El origen tiene distintos niveles de agregación (país, departamento, provincia, municipio/TIOC y, en algunos cuadros, urbano/rural). Filtraremos únicamente las filas **`Municipio/TIOC`**, tomando el total del territorio y los **datos de 2024**.

| Libro original | Hoja | Indicadores escogidos |
|---|---|---|
| `POBLACIÓN.xlsx` | `1` | Población total, hombres, mujeres |
| `VIVIENDA Y HOGAR.xlsx` | `22` | Promedio de personas por hogar |
| `SERVICIOS BÁSICOS.xlsx` | `1` | Viviendas con/sin electricidad |
| `SERVICIOS BÁSICOS.xlsx` | `7` | Personas con/sin agua mejorada |
| `SERVICIOS BÁSICOS.xlsx` | `11` | Personas con/sin saneamiento mejorado |

> Las columnas de **calidad de vivienda** (hoja 18) no forman parte de la extracción.

In [ ]:
# Números de columna Excel desde 1 (A=1, B=2, etc.).
# Verificados contra los encabezados de 2024 de los tabulados del INE.
SOURCES = {
    "poblacion": ("POBLACION.xlsx", "1", {
        "poblacion_total": 14,
        "poblacion_hombres": 15,
        "poblacion_mujeres": 16,
    }),
    "hogar": ("VIVIENDA Y HOGAR.xlsx", "22", {
        "promedio_personas_hogar": 10,
    }),
    "electricidad": ("SERVICIOS BASICOS.xlsx", "1", {
        "viviendas_electricidad_total": 14,
        "viviendas_con_electricidad": 15,
        "viviendas_sin_electricidad": 16,
    }),
    "agua": ("SERVICIOS BASICOS.xlsx", "7", {
        "personas_agua_total": 14,
        "personas_agua_mejorada": 15,
        "personas_sin_agua_mejorada": 16,
    }),
    "saneamiento": ("SERVICIOS BASICOS.xlsx", "11", {
        "personas_saneamiento_total": 14,
        "personas_saneamiento_mejorado": 15,
        "personas_sin_saneamiento_mejorado": 16,
    }),
}
GEO = ["departamento", "provincia", "municipio_tioc"]
print("Hojas a extraer:", len(SOURCES))
print("Libros distintos:", len({v[0] for v in SOURCES.values()}))

Hojas a extraer: 5
Libros distintos: 3


In [ ]:
# Extraer las cinco hojas directamente desde el ZIP, sin descomprimirlo.
# Cargar cada libro solamente una vez para no desperdiciar memoria/tiempo.
frames = {}
workbooks = {}

with ZipFile(ZIP_PATH) as zip_file:
    disponibles = set(zip_file.namelist())
    for nombre, (archivo, hoja, medidas) in SOURCES.items():
        if archivo not in disponibles:
            raise FileNotFoundError(f"Dentro del ZIP falta: {archivo}")
        if archivo not in workbooks:
            workbooks[archivo] = load_workbook(
                BytesIO(zip_file.read(archivo)), read_only=True, data_only=True
            )
        wb = workbooks[archivo]
        if hoja not in wb.sheetnames:
            raise KeyError(f"Falta la hoja {hoja} dentro de {archivo}")
        ws = wb[hoja]

        registros = []
        for fila in ws.iter_rows(min_row=9, values_only=True):
            # Se descartan totales País, Departamento, Provincia y subfilas Urbana/Rural.
            if not isinstance(fila[1], int) or fila[2] != "Municipio/TIOC":
                continue
            if fila[6] != fila[5]:
                continue
            registro = dict(zip(["n_orden", "nivel", *GEO, "area"], fila[1:7]))
            registro.update({col: fila[num_excel - 1] for col, num_excel in medidas.items()})
            registros.append(registro)

        df = pd.DataFrame(registros)
        if df.empty:
            raise ValueError(f"No se encontraron municipios/TIOC en {nombre}")
        if df.duplicated(GEO).any():
            raise ValueError(f"Claves geográficas duplicadas en {nombre}")
        frames[nombre] = df
        print(f"{nombre:13s}: {len(df):3d} filas y {df.shape[1]} columnas")

for wb in workbooks.values():
    wb.close()

# Control esperado para esta versión de los tabulados oficiales.
assert all(len(df) == 343 for df in frames.values()), "Revisar estructura/versión de los tabulados"

poblacion    : 343 filas y 9 columnas
hogar        : 343 filas y 7 columnas
electricidad : 343 filas y 9 columnas
agua         : 343 filas y 9 columnas
saneamiento  : 343 filas y 9 columnas


In [ ]:
# Integrar 3 libros (5 hojas) con uniones 1:1.
# Usamos la clave compuesta GEO, NO solo el nombre de municipio.
df_base = frames["poblacion"].copy()
for origen in ("hogar", "electricidad", "agua", "saneamiento"):
    df_derecha = frames[origen]
    auditoria = df_base[GEO + ["n_orden"]].merge(
        df_derecha[GEO + ["n_orden"]],
        on=GEO, how="outer", indicator=True, validate="one_to_one",
        suffixes=("_izq", "_der"),
    )
    if not auditoria["_merge"].eq("both").all():
        raise ValueError(f"Hay territorios sin correspondencia en {origen}")
    if not auditoria["n_orden_izq"].eq(auditoria["n_orden_der"]).all():
        raise ValueError(f"No coincide el número de orden de {origen}")

    df_base = df_base.merge(
        df_derecha.drop(columns=["n_orden", "nivel", "area"]),
        on=GEO, how="left", validate="one_to_one",
    )

print("Dimensiones antes de crear ratios:", df_base.shape)
display(df_base.head(3))

Dimensiones antes de crear ratios: (343, 19)


,n_orden,nivel,departamento,provincia,municipio_tioc,area,poblacion_total,poblacion_hombres,poblacion_mujeres,promedio_personas_hogar,viviendas_electricidad_total,viviendas_con_electricidad,viviendas_sin_electricidad,personas_agua_total,personas_agua_mejorada,personas_sin_agua_mejorada,personas_saneamiento_total,personas_saneamiento_mejorado,personas_sin_saneamiento_mejorado
0,10,Municipio/TIOC,Chuquisaca,Oropeza,Sucre,Sucre,296746,142122,154624,3.132085,92531,89370,3161,289815,272647,17168,289815,248278,41537
1,13,Municipio/TIOC,Chuquisaca,Oropeza,Yotala,Yotala,9256,4654,4602,2.938482,3056,2705,351,8980,7908,1072,8980,6789,2191
2,16,Municipio/TIOC,Chuquisaca,Oropeza,Poroma,Poroma,13649,6982,6667,2.923722,4654,2173,2481,13607,8942,4665,13607,2201,11406


In [17]:
df_base.head(100)

,n_orden,nivel,departamento,provincia,municipio_tioc,area,poblacion_total,poblacion_hombres,poblacion_mujeres,promedio_personas_hogar,...,personas_agua_total,personas_agua_mejorada,personas_sin_agua_mejorada,personas_saneamiento_total,personas_saneamiento_mejorado,personas_sin_saneamiento_mejorado,pct_mujeres,pct_electricidad,pct_agua_mejorada,pct_saneamiento_mejorado
0,10,Municipio/TIOC,Chuquisaca,Oropeza,Sucre,Sucre,296746,142122,154624,3.132085,...,289815,272647,17168,289815,248278,41537,52.106515,96.583848,94.076221,85.667754
1,13,Municipio/TIOC,Chuquisaca,Oropeza,Yotala,Yotala,9256,4654,4602,2.938482,...,8980,7908,1072,8980,6789,2191,49.719101,88.514398,88.062361,75.601336
2,16,Municipio/TIOC,Chuquisaca,Oropeza,Poroma,Poroma,13649,6982,6667,2.923722,...,13607,8942,4665,13607,2201,11406,48.846069,46.691018,65.716175,16.175498
3,22,Municipio/TIOC,Chuquisaca,Azurduy,Azurduy,Azurduy,7903,4004,3899,3.199592,...,7839,5493,2346,7839,2851,4988,49.335695,55.306122,70.072713,36.369435
4,25,Municipio/TIOC,Chuquisaca,Azurduy,Tarvita,Tarvita,16812,8268,8544,3.635276,...,16715,13280,3435,16715,6520,10195,50.820842,68.334058,79.449596,39.006880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,364,Municipio/TIOC,La Paz,Aroma,Ayo Ayo,Ayo Ayo,11258,5778,5480,2.345015,...,11242,7104,4138,11242,4025,7217,48.676497,67.167292,63.191603,35.803238
96,367,Municipio/TIOC,La Paz,Aroma,Calamarca,Calamarca,15327,7744,7583,2.789281,...,15249,12509,2740,15249,8610,6639,49.474783,80.885312,82.031609,56.462719
97,370,Municipio/TIOC,La Paz,Aroma,Patacamaya,Patacamaya,25148,12564,12584,2.617096,...,24462,21182,3280,24462,13466,10996,50.039765,84.604686,86.591448,55.048647
98,373,Municipio/TIOC,La Paz,Aroma,Colquencha,Colquencha,12490,6312,6178,2.783036,...,12468,10493,1975,12468,3970,8498,49.463571,76.294643,84.159448,31.841514


In [ ]:
#  Crear métricas calculadas, respetando el denominador de cada fuente.
RATIOS = {
    "pct_mujeres": ("poblacion_mujeres", "poblacion_total"),
    "pct_electricidad": ("viviendas_con_electricidad", "viviendas_electricidad_total"),
    "pct_agua_mejorada": ("personas_agua_mejorada", "personas_agua_total"),
    "pct_saneamiento_mejorado": (
        "personas_saneamiento_mejorado", "personas_saneamiento_total"
    ),
}
for salida, (numerador, denominador) in RATIOS.items():
    den = df_base[denominador].replace(0, np.nan)
    df_base[salida] = 100 * df_base[numerador] / den

print("Tamaño de la tabla base:", df_base.shape)
display(df_base[[*GEO, "poblacion_total", "promedio_personas_hogar", *RATIOS]].head())

Tamaño de la tabla base: (343, 23)


,departamento,provincia,municipio_tioc,poblacion_total,promedio_personas_hogar,pct_mujeres,pct_electricidad,pct_agua_mejorada,pct_saneamiento_mejorado
0,Chuquisaca,Oropeza,Sucre,296746,3.132085,52.106515,96.583848,94.076221,85.667754
1,Chuquisaca,Oropeza,Yotala,9256,2.938482,49.719101,88.514398,88.062361,75.601336
2,Chuquisaca,Oropeza,Poroma,13649,2.923722,48.846069,46.691018,65.716175,16.175498
3,Chuquisaca,Azurduy,Azurduy,7903,3.199592,49.335695,55.306122,70.072713,36.369435
4,Chuquisaca,Azurduy,Tarvita,16812,3.635276,50.820842,68.334058,79.449596,39.006880
